<h1> CNN-Based Classification Model Tutorial </h1>
<h5> <p style="font-size:90% ; font-family:arial"> (1) options.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (2) pipeline.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (3) networks.py </p> </h5> 
<h5> <p style="font-size:120% ; line-height:50% ; color:blue ; font-weight:bold">  (4) train.py </p> </h5> 

In [ ]:
import os
import sys

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    sys.path.append("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
    os.chdir("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
except ModuleNotFoundError:
    print("Not in colab, continue")

위 내용은 앞에서 다뤘습니다. <br>

In [ ]:
import time
import torch
import numpy as np

내/외부 라이브러리를 불러옵니다. <br>

In [ ]:
from options import TrainOptions
opt = TrainOptions().parse()

TrainOptions를 불러와 선언합니다.

In [ ]:
from networks import define_network, define_criterion, define_optimizer
from pipeline import define_dataset
from utils import fix_seed, get_num_params

networks.py 에서 define_network, define_criterion, define_optimizer를, <br> <br>
pipeline.py 에서 define_dataset을, <br> <br>
utils.py 에서 fix_seed, get_num_params를 불러옵니다. <br> <br>


In [ ]:
fix_seed(opt.seed)

랜덤시드를 고정합니다. 시드 값은 option에 정의되어 있습니다.

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

device를 정의 합니다. <br> <br>
현재 실행 환경에 맞는 device가 출력됩니다. <br>

In [ ]:
network = define_network(opt).to(device)
criterion = define_criterion(opt).to(device)
optimizer = define_optimizer(network, opt)
print(network)
print(f"Number of parameters : {get_num_params(network)}")
print(criterion)
print(optimizer)

network. criterion, optimizer를 선언합니다. <br> <br>
network를 device로 옮겨줍니다. <br>

In [ ]:
dataset, dataloader = define_dataset(opt)
print(len(dataset), len(dataloader))

dataset, dataloader를 선언합니다. <br>

In [ ]:
save_dir = os.path.join(opt.save_root, opt.name)
os.makedirs(save_dir, exist_ok=True)

학습 결과를 저장할 경로를 선언하고 경로에 폴더를 생성합니다. <br>

In [ ]:
dataloader = iter(dataloader)

data = next(dataloader)
image, label = data

임의의 데이터를 한 쌍 불러옵니다. <br>

In [ ]:
image = image.to(device)
label = label.to(device)

데이터를 device로 옮깁니다.

In [ ]:
output = network(image)

image를 네트워크에 입력하여 결과를 하나 출력합니다.

In [ ]:
optimizer.zero_grad()

optimizer에 저장된 gradient를 모두 지워줍니다. <br> <br>
이 과정은 매번 반복해주어야 합니다. <br>

In [ ]:
loss = criterion(output, label)
print(loss)

모델이 출력한 결과와 실제 정답 (label)를 이용하여 loss를 계산합니다. <br>

In [ ]:
loss.backward()

계산된 loss로부터 backpropagation을 수행합니다. <br>

In [ ]:
optimizer.step()

backpropagation된 gradient를 이용하여 모델 업데이트를 수행합니다. <br>

In [ ]:
image, label = next(dataloader)

image = image.to(device)
label = label.to(device)

optimizer.zero_grad()
output = network(image)
loss = criterion(output, label)
loss.backward()
optimizer.step()

위의 과정을 요약하면, <br> <br>
(1) dataloader 로부터 하나의 배치 데이터를 받아서 <br>
(2) 데이터를 device에 올리고 <br>
(3) optimizer에 저장된 gradient를 초기화하고 <br>
(4) 데이터를 이용하여 모델이 출력을 만들고 <br>
(5) loss를 계산하고 <br>
(6) loss를 backpropagation하여 gradient를 계산하고 <br>
(7) optimizer가 모델을 업데이트하고 <br>
(8) 이 작업을 반복 <br> <br>
이 과정은 한 묶음으로 실행됩니다. <br>

In [ ]:
## 학습 시작
network.train()
iters = 0
epochs = 0
losses = []
t0 = time.time()

while epochs < opt.num_epochs:

    for idx, (image, label) in enumerate(dataloader):
        image = image.to(device)
        label = label.to(device)

        # 실제 학습이 이루어지는 부분
        optimizer.zero_grad()
        output = network(image)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        # 실제 학습이 이루어지는 부분

        losses.append(loss.item())            
        iters += 1

        if iters % 100 == 0:
            print(f"Epoch [{epochs}/{opt.num_epochs}], Step [{iters}], Loss: {np.mean(losses):.4f}, Time: {time.time()-t0:.4f}")
            losses = []
            t0 = time.time()

    epochs += 1

    if epochs % 5 == 0 :
        state_network = network.state_dict()
        state_optimizer = optimizer.state_dict()
        state = {"network": state_network, "optimizer": state_optimizer, "epoch": epochs}
        torch.save(state, f"{save_dir}/model_{epochs:04d}.pt")

state_network = network.state_dict()
state_optimizer = optimizer.state_dict()
state = {"network": state_network, "optimizer": state_optimizer, "epoch": epochs}
torch.save(state, f"{save_dir}/model_final.pt")

훈련 과정을 epoch 단위로 수행하는 코드입니다. <br> <br>
특정 간격마다 결과를 출력하고 <br> <br>
특정 epoch 마다 모델을 저장하도록 되어 있습니다. <br>